# Chapter 3 — Pretraining, generation, SFT, and GRPO
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch03_pretrain_sft_grpo.ipynb)

A T4-sized smoke-test version of the chapter. The goal is to verify the complete learning path without spending hours on full training.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
device='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(device)

## 1. Tiny causal language model
For this chapter we use PyTorch's TransformerEncoder only to keep focus on the training algorithms. Chapter 2 contains the from-scratch attention implementation.

In [ ]:
class TinyLM(nn.Module):
    def __init__(self,vocab=128,d=128,nhead=4,layers=2,ctx=128):
        super().__init__(); self.ctx=ctx
        self.tok=nn.Embedding(vocab,d); self.pos=nn.Embedding(ctx,d)
        layer=nn.TransformerEncoderLayer(d,nhead,4*d,batch_first=True,norm_first=True)
        self.tr=nn.TransformerEncoder(layer,layers); self.ln=nn.LayerNorm(d); self.head=nn.Linear(d,vocab,bias=False)
        self.head.weight=self.tok.weight
    def forward(self,x):
        T=x.size(1); h=self.tok(x)+self.pos(torch.arange(T,device=x.device))
        mask=torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool),1)
        return self.head(self.ln(self.tr(h,mask=mask)))

model=TinyLM().to(device); print(sum(p.numel() for p in model.parameters()))

## 2. Pretraining: next-token prediction

In [ ]:
data=(torch.arange(20000)%97).long()
opt=torch.optim.AdamW(model.parameters(),lr=3e-4)
for step in range(60):
    start=torch.randint(0,len(data)-65,(16,))
    x=torch.stack([data[s:s+64] for s in start]).to(device)
    y=torch.stack([data[s+1:s+65] for s in start]).to(device)
    loss=F.cross_entropy(model(x).reshape(-1,128),y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step%10==0: print(step,float(loss))

## 3. Autoregressive generation

In [ ]:
@torch.no_grad()
def generate(model,idx,new_tokens=20,temperature=1.0):
    for _ in range(new_tokens):
        logits=model(idx[:,-model.ctx:])[:,-1]/temperature
        p=F.softmax(logits,dim=-1); nxt=torch.multinomial(p,1)
        idx=torch.cat([idx,nxt],dim=1)
    return idx
print(generate(model,torch.tensor([[1,2,3]],device=device),20))

## 4. SFT response-only loss
Prompt positions receive target `-100`, so cross entropy ignores them.

In [ ]:
x=torch.randint(0,128,(8,48),device=device)
targets=x.roll(-1,1); targets[:,:24]=-100
logits=model(x)
sft_loss=F.cross_entropy(logits.reshape(-1,128),targets.reshape(-1),ignore_index=-100)
opt.zero_grad(); sft_loss.backward(); opt.step()
print('SFT loss:',float(sft_loss))

## 5. GRPO core objective
This cell isolates the chapter's central idea: normalize rewards within a group, form advantages, and optimize a clipped policy ratio with a KL-style regularizer.

In [ ]:
rewards=torch.tensor([1.0,0.2,0.7,-0.1],device=device)
adv=(rewards-rewards.mean())/(rewards.std()+1e-6)
old_logp=torch.tensor([-1.2,-1.0,-1.4,-1.1],device=device)
new_logp=torch.tensor([-1.1,-1.1,-1.2,-1.0],device=device,requires_grad=True)
ratio=(new_logp-old_logp).exp(); eps=0.2
policy=torch.minimum(ratio*adv,ratio.clamp(1-eps,1+eps)*adv)
loss=-policy.mean(); loss.backward()
print('advantages=',adv.tolist()); print('GRPO surrogate loss=',float(loss))

## T4 note
The official chapter includes longer pretraining/SFT/GRPO runs. On T4, keep the architecture but use this notebook first as a forward/backward and loss-decrease check; increase steps only when needed.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch03